# 地域幸福度（Well-Being）指標 CSV自動ダウンロードプログラム

デジタル庁の「地域幸福度（Well-Being）指標ダッシュボード」から、全年度・全都道府県・全市区町村のCSVデータを全自動で取得・保存するスクリプトです。

## 概要
* **対象サイト**: https://well-being.digital.go.jp/dashboard/
* **取得データ**: 全年度 × 47都道府県 × 全市区町村の個別CSVファイル（幸福度・生活満足度 / カテゴリー別 / 主観）
* **保存先**: `./downloaded_csv/` （実行ディレクトリ直下）

## 主な処理の流れ
1. ダッシュボードを開き、選択可能な「年度」を自動検出
2. 年度 ＞ 都道府県 ＞ 市区町村 の順にドロップダウンを切り替え
3. ページ下部までスクロールし、「CSVデータをダウンロード」ボタンを検出して保存
4. すでに保存済みのファイルが存在する場合は自動スキップ

## 必要なライブラリ
* `playwright`
* `asyncio`

In [2]:
import asyncio
import os
from playwright.async_api import async_playwright

SAVE_DIR = "./downloaded_csv"
os.makedirs(SAVE_DIR, exist_ok=True)

# 除外ワード（ヘッダーメニュー等の誤判定防止）
EXCLUDE_WORDS = [
    "RAIDA",
    "RESAS",
    "ダッシュボード",
    "指標について知る",
    "使いこなす",
    "個別調査をする",
    "ホーム",
    "ログイン",
    "利用規約",
    "プライバシーポリシー",
    "ウェブアクセシビリティ",
    "メニュー",
    "閉じる",
    "都道府県を選択",
    "市区町村を選択",
    "調査種別を選択",
    "年度を選択",
]

PREFECTURES = [
    "北海道",
    "青森県",
    "岩手県",
    "宮城県",
    "秋田県",
    "山形県",
    "福島県",
    "茨城県",
    "栃木県",
    "群馬県",
    "埼玉県",
    "千葉県",
    "東京都",
    "神奈川県",
    "新潟県",
    "富山県",
    "石川県",
    "福井県",
    "山梨県",
    "長野県",
    "岐阜県",
    "静岡県",
    "愛知県",
    "三重県",
    "滋賀県",
    "京都府",
    "大阪府",
    "兵庫県",
    "奈良県",
    "和歌山県",
    "鳥取県",
    "島根県",
    "岡山県",
    "広島県",
    "山口県",
    "徳島県",
    "香川県",
    "愛媛県",
    "高知県",
    "福岡県",
    "佐賀県",
    "長崎県",
    "熊本県",
    "大分県",
    "宮崎県",
    "鹿児島県",
    "沖縄県",
]


async def open_select_box(page, label_text):
    """「年度を選択」「都道府県を選択」「市区町村を選択」の枠を開く"""
    try:
        label_el = page.get_by_text(label_text, exact=True).first
        parent_box = label_el.locator("xpath=..")
        box = parent_box.locator("div, button, input").last
        await box.click(force=True)
        await asyncio.sleep(1)
        return True
    except Exception as e:
        print(f"⚠️ 枠 [{label_text}] の展開失敗: {e}")
        return False


async def get_dropdown_options(page):
    """ドロップダウン内の選択肢リストを取得"""
    await asyncio.sleep(0.5)
    return await page.evaluate(
        """(excludes) => {
        const nodes = Array.from(document.querySelectorAll('[role="option"], [class*="option"], [class*="Option"], [class*="menu"] li'));
        const list = [];
        for (const el of nodes) {
            const txt = el.textContent.trim();
            const rect = el.getBoundingClientRect();
            if (rect.width > 0 && rect.height > 0 && txt) {
                const isExcluded = excludes.some(ex => txt.includes(ex));
                if (!isExcluded && !list.includes(txt)) {
                    list.push(txt);
                }
            }
        }
        return list;
    }""",
        EXCLUDE_WORDS,
    )


async def select_option(page, target_text):
    """ドロップダウン内の項目を選択"""
    try:
        option_loc = page.locator(
            f'[role="option"]:has-text("{target_text}"), div[class*="option"]:has-text("{target_text}")'
        ).first
        if await option_loc.is_visible(timeout=2000):
            await option_loc.click(force=True)
        else:
            await page.get_by_text(target_text, exact=True).first.click(
                timeout=3000, force=True
            )
        await asyncio.sleep(1.5)
        return True
    except Exception as e:
        print(f"❌ 選択肢 [{target_text}] クリック失敗: {e}")
        return False


async def download_all_wellbeing_data():
    async with async_playwright() as p:
        browser = await p.chromium.launch(headless=False)
        context = await browser.new_context(accept_downloads=True)
        page = await context.new_page()

        print("🌐 ダッシュボードを開いています...")
        await page.goto(
            "https://well-being.digital.go.jp/dashboard/",
            wait_until="networkidle",
        )

        await page.get_by_text("表示中のデータ選択条件").wait_for(
            timeout=30000
        )
        await asyncio.sleep(3)

        # --------------------------------------------------
        # 1. 選択可能な「年度」一覧を自動取得
        # --------------------------------------------------
        years = []
        if await open_select_box(page, "年度を選択"):
            years = await get_dropdown_options(page)
            await page.mouse.click(10, 10)  # 閉じる
            await asyncio.sleep(0.8)

        if not years:
            print(
                "⚠️ 年度の自動取得に失敗したため、2026年度のみで実行します。"
            )
            years = ["2026年度"]

        print(f"🗓️ 検出された年度 ({len(years)}件): {years}")

        # --------------------------------------------------
        # 2. 年度 ＞ 都道府県 ＞ 市区町村 のトリプルループ
        # --------------------------------------------------
        for year in years:
            print(f"\n========================================")
            print(f"🗓️ 【年度: {year}】 の処理開始")
            print(f"========================================")

            # 年度を変更
            if await open_select_box(page, "年度を選択"):
                await select_option(page, year)

            # ファイル名用に年度の文字列を整形 (例: 2026年度)
            short_year = (
                year.split("版")[0] if "版" in year else year.replace(" ", "")
            )

            for pref in PREFECTURES:
                print(f"\n--- [{short_year}] 📍 【{pref}】 の処理開始 ---")

                try:
                    # 都道府県を選択
                    if await open_select_box(page, "都道府県を選択"):
                        await select_option(page, pref)

                    # 市区町村一覧を取得
                    cities = []
                    if await open_select_box(page, "市区町村を選択"):
                        cities = await get_dropdown_options(page)
                        await page.mouse.click(10, 10)
                        await asyncio.sleep(0.8)

                    print(
                        f"  [{pref}] 検出された市区町村: {len(cities)} 件"
                    )

                    if not cities:
                        print(
                            f"  ⚠️ 市区町村一覧が取得できませんでした。スキップします。"
                        )
                        continue

                    for city in cities:
                        filename = f"{short_year}_{pref}_{city}.csv"
                        save_path = os.path.join(SAVE_DIR, filename)

                        # 取得済みファイルはスキップ
                        if os.path.exists(save_path):
                            print(
                                f"  ⏩ スキップ (完了済み): {filename}"
                            )
                            continue

                        print(
                            f"  📥 ダウンロード中: {short_year} {pref} {city}"
                        )

                        try:
                            # 市区町村を選択
                            if await open_select_box(
                                page, "市区町村を選択"
                            ):
                                await select_option(page, city)

                            # ページ下部へスクロールしてデータ読込を待つ
                            await page.evaluate(
                                "window.scrollTo(0, document.body.scrollHeight)"
                            )
                            await asyncio.sleep(2)

                            # CSVボタンの特定と待機
                            csv_target = page.get_by_text(
                                "CSVデータをダウンロード"
                            ).last
                            try:
                                await csv_target.wait_for(
                                    state="visible", timeout=10000
                                )
                            except Exception:
                                csv_target = page.locator(
                                    '*:has-text("CSVデータをダウンロード")'
                                ).last

                            await csv_target.scroll_into_view_if_needed()

                            # ダウンロード実行
                            async with page.expect_download(
                                timeout=15000
                            ) as download_info:
                                try:
                                    await csv_target.click(force=True)
                                except Exception:
                                    await page.evaluate("""() => {
                                        const all = Array.from(document.querySelectorAll('*'));
                                        const btn = all.reverse().find(el => el.textContent.includes('CSVデータをダウンロード') && el.children.length === 0);
                                        if (btn) btn.click();
                                    }""")

                            download = await download_info.value
                            await download.save_as(save_path)
                            print(f"     └ 成功保存: {save_path}")

                            await asyncio.sleep(1)

                        except Exception as e:
                            print(f"     └ ❌ エラー ({city}): {e}")

                except Exception as e:
                    print(
                        f"❌ [{short_year} - {pref}] 処理エラー: {e}"
                    )

        await browser.close()
        print("\n🎉 全年度・全データのダウンロード処理がすべて完了しました！")


# 実行
await download_all_wellbeing_data()

🌐 ダッシュボードを開いています...


CancelledError: 

In [15]:
import os

print(os.path.abspath("./downloaded_csv"))

/home/nishimura/wakayama_competition/nishimura/downloaded_csv
